# Make figures for the SCA comparison

Rainey Aberle (rainey.aberle@usace.army.mil)

Snow-Informed Reservoir Operations (SIRO)

USACE-ERDC-CRREL

June 2026

In [ ]:
import os
from glob import glob
import xarray as xr
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from tqdm import tqdm

plt.rcParams.update({'font.size': 12, 'font.sans-serif': 'Verdana'})

# I/O
BASE_DIR = "/Users/rdcrlrka/Research/SIRO/MCS_SCA/"
OUT_DIR = os.path.join(BASE_DIR, "SCA_comparison_results")
FIG_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Threshold params
FSCA_THRESHOLD = 0.5
TREE_MAX = 0.5
VALID_MIN = 0.5

# Model names
MODEL_NAMES = ["HMS-EB", "HMS-TI", "iSnobal", "SnowModel"]

# Consolidated result files
PSS_MOSAIC_FILES    =  sorted(glob(os.path.join(BASE_DIR, "PSS_image_mosaics_clipped", "*.tif")))
FSCA_PSS_FILE       =  os.path.join(OUT_DIR, "fSCA_100m_PlanetScope.nc")
PSS_TOTALS_FILE     =  os.path.join(OUT_DIR, "SCA_totals_100m_PlanetScope.nc")
SCA_MODELED_FILE    =  os.path.join(OUT_DIR, "compiled_SCA_totals_100m_modeled.nc")
CM_FILE             =  os.path.join(OUT_DIR, "compiled_confusion_matrices.nc")
RECALL_FILE         =  os.path.join(OUT_DIR, "compiled_recall_binned_elev_aspect.nc")

# One gridded file per model
MODEL_GRID_FILES = {
    m: os.path.join(OUT_DIR, f"SCA_100m_gridded_{m}.nc") for m in MODEL_NAMES
}

print(f"Located {len(PSS_MOSAIC_FILES)} PSS mosaics")
for file in [FSCA_PSS_FILE, PSS_TOTALS_FILE, SCA_MODELED_FILE, CM_FILE, RECALL_FILE]:
    if not os.path.exists(file):
        print("Could not locate file:", file)
for m, f in MODEL_GRID_FILES.items():
    if not os.path.exists(f):
        print(f"Could not locate gridded file for {m}:", f)


## Define some color palettes

In [ ]:
sca_cmap = matplotlib.colors.ListedColormap([
    "#e5e5e5",   # 0.0  -> no snow
    "#d1b176",   # 0.5  -> uncertain
    "#137C8D"   # 1.0  -> snow
])
sca_cmap.set_bad("#ffffff")  # NaN (unobserved)
# Boundaries centered so 0, 0.5, 1 fall in the middle of each color band
sca_cmap_norm = matplotlib.colors.BoundaryNorm([-0.25, 0.25, 0.75, 1.25], sca_cmap.N)
sca_cmap

In [ ]:
land_cover_cmap = matplotlib.colors.ListedColormap(["#e5e5e5", "#14c7e2", "#2b6c35"])
land_cover_cmap

In [ ]:
ds_colors_dict = {
    "PlanetScope":  "#000000",
    "HMS-EB":       "#7D2B54",
    "HMS-TI":       "#BF6B78",
    "iSnobal":      "#97CAEA",
    "SnowModel":    "#37753B"
}
ds_colors = list(ds_colors_dict.values())
ds_cmap = matplotlib.colors.ListedColormap(ds_colors)
ds_cmap


In [ ]:
recall_cmap = plt.cm.cividis
recall_cmap_norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
recall_cmap

## Map timeseries GIF

In [ ]:
TASK = 2
SWE_THRESHOLD_PLOT = 0.0
TARGET_RES = 100
PIXEL_AREA = TARGET_RES ** 2
NODATA_VAL = 255

# --- Open the gridded datasets ---
pss_fsca_ds = xr.open_dataset(FSCA_PSS_FILE)
model_grid_das = {
    m: xr.open_dataset(MODEL_GRID_FILES[m])["SCA"] for m in MODEL_NAMES
}

pss_time = pd.to_datetime(pss_fsca_ds["time"].values)


def get_pss_series():
    """
    Raw PlanetScope SCA totals over the full target grid (no comparison mask),
    so they line up with the raw model maps.

    best  = snow area                       = sum(snow_frac) * pixel_area
    lower = certain snow only               = best (uncertain area assumed no-snow)
    upper = snow + all uncertain area       = best + sum(1 - certain_frac) * pixel_area

    Uncertain area = the part of each observed cell whose snow status is not
    certain (trees + partially/edge-classified) = (1 - certain_frac), counted
    only where the cell was observed at all.
    """
    snow_frac = pss_fsca_ds["snow_frac"]
    certain_frac = pss_fsca_ds["certain_frac"]
    valid_frac = pss_fsca_ds["valid_frac"]

    observed = valid_frac > 0
    uncertain_frac = (1.0 - certain_frac).clip(min=0.0).where(observed, 0.0)

    best = (snow_frac.where(observed, 0.0).sum(dim=["x", "y"]) * PIXEL_AREA).values / 1e6
    uncertain_area = (uncertain_frac.sum(dim=["x", "y"]) * PIXEL_AREA).values / 1e6

    lower = best                     # uncertain -> no snow
    upper = best + uncertain_area    # uncertain -> snow
    return pss_time, best, lower, upper


def get_model_series(model, task=TASK):
    """
    RAW model SCA totals over the FULL target grid (no comparison mask), matching
    the raw model map panels. min/mean/max are taken across SWE thresholds.
    """
    da = model_grid_das[model].sel(task=task)          # dims: (SWE_threshold_m, time, y, x)
    da = xr.where(da == NODATA_VAL, np.nan, da)         # drop nodata

    # snow area per (SWE_threshold_m, time) = count of snow cells * pixel area
    snow_area = (da == 1).sum(dim=["x", "y"]) * PIXEL_AREA   # (SWE_threshold_m, time)
    snow_area_km2 = snow_area / 1e6

    t = pd.to_datetime(da["time"].values)
    mean_km2 = snow_area_km2.mean(dim="SWE_threshold_m", skipna=True).values
    min_km2 = snow_area_km2.min(dim="SWE_threshold_m", skipna=True).values
    max_km2 = snow_area_km2.max(dim="SWE_threshold_m", skipna=True).values
    return t, mean_km2, min_km2, max_km2

In [ ]:
gif_out_dir = os.path.join(FIG_DIR, "SCA_gif")
os.makedirs(gif_out_dir, exist_ok=True)

def process_date(
        date, pss_mosaic_file, pss_fsca_ds, model_grid_das,
        gif_out_dir, sca_cmap,
        ):

    dt = np.datetime64(date)
    fig_file = os.path.join(gif_out_dir, f"{date}_SCA_comparison_Task{TASK}.png")
    if os.path.exists(fig_file):
        return

    with rxr.open_rasterio(pss_mosaic_file, masked=True, chunks={'x': 2048, 'y': 2048}).squeeze() as pss_mosaic:
        pss_mosaic = pss_mosaic / 1e4

        # select PlanetScope fractional fields at nearest time
        pss_sca_samp = pss_fsca_ds.sel(time=dt, method='nearest')

        # Preprocess PlanetScope into snow (1), no snow (0), and uncertain (0.5) categories
        comp_mask = (
            (pss_sca_samp.valid_frac >= 0.5)
            & (pss_sca_samp.tree_frac <= 0.5)
            & (pss_sca_samp.certain_frac > 0)
        )
        # observed but uncertain (e.g. too many trees)
        observed = ~np.isnan(pss_sca_samp.fsca) | (pss_sca_samp.valid_frac > 0)
        uncertain = observed & (~comp_mask)
        pss_sca_samp_3class = xr.full_like(pss_sca_samp.fsca, np.nan)
        # confidently classified cells -> 0 (no snow) or 1 (snow)
        pss_sca_samp_3class = xr.where(comp_mask & (pss_sca_samp.fsca >= 0.5), 1.0, pss_sca_samp_3class)
        pss_sca_samp_3class = xr.where(comp_mask & (pss_sca_samp.fsca < 0.5), 0.0, pss_sca_samp_3class)
        # Observed-but-uncertain cells -> 0.5
        pss_sca_samp_3class = xr.where(uncertain, 0.5, pss_sca_samp_3class)

        # Slice each model's gridded SCA at (task, SWE threshold, nearest time)
        model_samps = {}
        for model in MODEL_NAMES:
            da = model_grid_das[model].sel(
                task=TASK, SWE_threshold_m=SWE_THRESHOLD_PLOT
            ).sel(time=dt, method='nearest')
            model_samps[model] = da

        # Plot
        gs = matplotlib.gridspec.GridSpec(3, 3, height_ratios=[2, 2, 1])
        fig = plt.figure(figsize=(12, 12))
        ax = [
            fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2]),
            fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[1, 2]),
            fig.add_subplot(gs[2, :]),
        ]

        # RGB
        ax[0].imshow(
            np.dstack([pss_mosaic.isel(band=2), pss_mosaic.isel(band=1), pss_mosaic.isel(band=0)]),
            clim=(0, 1),
            extent=(min(pss_mosaic.x) / 1e3, max(pss_mosaic.x) / 1e3,
                    min(pss_mosaic.y) / 1e3, max(pss_mosaic.y) / 1e3)
        )
        ax[0].set_title("PlanetScope RGB")

        # SCA maps
        sca_maps = [
            (pss_sca_samp_3class, "PlanetScope SCA"),
            (model_samps["iSnobal"], "iSnobal SCA"),
            (model_samps["SnowModel"], "SnowModel SCA"),
            (model_samps["HMS-EB"], "HMS-EB SCA"),
            (model_samps["HMS-TI"], "HMS-TI SCA"),
        ]
        im = None
        for (sca_map, title), axis in zip(sca_maps, ax[1:]):
            # mask nodata values
            sca_map = xr.where(sca_map != 255, sca_map, np.nan)

            im = axis.imshow(
                sca_map.data, cmap=sca_cmap, norm=sca_cmap_norm,
                extent=(
                    min(sca_map.x) / 1e3, max(sca_map.x) / 1e3,
                    min(sca_map.y) / 1e3, max(sca_map.y) / 1e3
                    )
            )
            axis.set_title(title)

        # SCA time series
        # PlanetScope
        t_pss, best, lower, upper = get_pss_series()
        pss_mask = t_pss <= dt
        ax[6].fill_between(
            t_pss[pss_mask], lower[pss_mask], upper[pss_mask],
            color=ds_colors_dict["PlanetScope"], alpha=0.2
        )
        ax[6].plot(
            t_pss[pss_mask], best[pss_mask],
            '.-', color=ds_colors_dict["PlanetScope"], label="PlanetScope"
        )

        # Models (min/mean/max across SWE thresholds)
        for model in MODEL_NAMES:
            t, mean_km2, min_km2, max_km2 = get_model_series(model, task=TASK)
            m_mask = t <= dt
            ax[6].fill_between(
                t[m_mask], min_km2[m_mask], max_km2[m_mask],
                color=ds_colors_dict[model], alpha=0.3
            )
            ax[6].plot(
                t[m_mask], mean_km2[m_mask],
                '.-', color=ds_colors_dict[model], label=model
            )

        ax[6].set_xlim(np.datetime64("2022-09-15"), np.datetime64("2025-07-15"))
        ax[6].set_ylim(-50, 1350)
        ax[6].set_xlabel("Date")
        ax[6].set_ylabel("Snow-covered area [km$^2$]")
        ax[6].legend(loc='center right', bbox_to_anchor=[1.05, 0.4, 0.2, 0.2])
        ax[6].grid(alpha=0.3)

        # axes labels
        for axis in [ax[0], ax[3]]:
            axis.set_ylabel("Northing [km]")
        for axis in ax[3:6]:
            axis.set_xlabel("Easting [km]")

        # colorbar
        if im is not None:
            cax = fig.add_axes([0.93, 0.5, 0.02, 0.2])
            cbar = fig.colorbar(im, cax=cax, ticks=[0.0, 0.5, 1.0])
            cbar.set_ticklabels(["No snow", "Uncertain", "Snow"])

        fig.suptitle(date)

        # Save to file
        fig.savefig(fig_file, dpi=250, bbox_inches='tight')
        plt.close()


# Iterate over dates
dates = [os.path.basename(f).split('_')[0] for f in PSS_MOSAIC_FILES]
for date, pss_mosaic_file in tqdm(list(zip(dates, PSS_MOSAIC_FILES))):
    process_date(
        date, pss_mosaic_file, pss_fsca_ds, model_grid_das,
        gif_out_dir, sca_cmap
    )

# Clean up
pss_fsca_ds.close()
for da in model_grid_das.values():
    da.close()

## Example PlanetScope SCA calculation

In [ ]:
fig_file = os.path.join(FIG_DIR, "PSS_SCA_example.png")

# Load files
pss_mosaic_file = PSS_MOSAIC_FILES[0]
pss_sca_file = sorted(glob(os.path.join(BASE_DIR, "PSS_SCA", "*.tif")))[0]

with (
    rxr.open_rasterio(pss_mosaic_file, masked=True).squeeze() as pss_mosaic,
    rxr.open_rasterio(pss_sca_file, masked=True).squeeze() as pss_sca,
    xr.open_dataset(FSCA_PSS_FILE) as pss_fsca,
):
    # Preprocess
    pss_mosaic = pss_mosaic / 1e4
    pss_NDSI = ((pss_mosaic.isel(band=1) - pss_mosaic.isel(band=3))
                / (pss_mosaic.isel(band=1) + pss_mosaic.isel(band=3)))

    pss_fsca_samp = pss_fsca.isel(time=0)
    comp_mask = (
        (pss_fsca_samp.valid_frac >= 0.5)
        & (pss_fsca_samp.tree_frac <= 0.5)
        & (pss_fsca_samp.certain_frac > 0)
    )
    # observed but uncertain (e.g. too many trees)
    observed = ~np.isnan(pss_fsca_samp.fsca) | (pss_fsca_samp.valid_frac > 0)
    uncertain = observed & (~comp_mask)
    pss_fsca_samp_3class = xr.full_like(pss_fsca_samp.fsca, np.nan)
    # confidently classified cells -> 0 (no snow) or 1 (snow)
    pss_fsca_samp_3class = xr.where(comp_mask & (pss_fsca_samp.fsca >= 0.5), 1.0, pss_fsca_samp_3class)
    pss_fsca_samp_3class = xr.where(comp_mask & (pss_fsca_samp.fsca < 0.5), 0.0, pss_fsca_samp_3class)
    # observed-but-uncertain cells -> 0.5
    pss_fsca_samp_3class = xr.where(uncertain, 0.5, pss_fsca_samp_3class)


    # Helper for clean float extents
    def ext(da):
        return (float(da.x.min()), float(da.x.max()),
                float(da.y.min()), float(da.y.max()))

    # Helper for imshow and colorbar
    def plot_img(
        ax, img, title, cmap, clim, extent,
        cbar_ticks=None, cbar_ticklabels=None, cbar_label=None
        ):
        im = ax.imshow(img, cmap=cmap, clim=clim, extent=extent)
        ax.set_title(title)
        if cmap is not None:
            cax = fig.add_axes([ax.get_position().x0,
                                ax.get_position().y0 - 0.05,
                                ax.get_position().width, 0.02])
            cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
            if cbar_ticks is not None:
                cbar.set_ticks(cbar_ticks)
            if cbar_ticklabels is not None:
                cbar.set_ticklabels(cbar_ticklabels)
            if cbar_label is not None:
                cbar.set_label(cbar_label)
        return im

    # Prepare images and settings
    images = [
        # RGB
        dict(
            img=np.dstack([
                pss_mosaic.isel(band=2),
                pss_mosaic.isel(band=1),
                pss_mosaic.isel(band=0)
                ]),
            title="RGB", cmap=None, clim=None,
            extent=ext(pss_mosaic),
        ),
        # NDSI
        dict(
            img=pss_NDSI.data, title="NDSI$_{mod}$", cmap=plt.cm.BrBG,
            clim=(-1, 1), extent=ext(pss_NDSI), cbar_ticks=[-1, 0, 1],
        ),
        # Land cover mask (native 3-5 m): 0=no snow, 1=snow, 2=trees
        dict(
            img=pss_sca.data, title="Land cover mask, native grid",
            cmap=land_cover_cmap, clim=(-0.25, 1.25), extent=ext(pss_sca),
            cbar_ticks=[0.0, 0.5, 1.0],
            cbar_ticklabels=["No snow", "Snow", "Trees"],
        ),
        # (intentionally blank panel — visual spacer for the flow arrows)
        dict(img=None, title="", cmap=None, clim=None, extent=None),
        # Binary SCA
        dict(
            img=pss_fsca_samp_3class.data,
            title="Binary SCA, 100 m grid\n(> 50% fSCA, < 50% trees)",
            cmap=sca_cmap, clim=(-0.25, 1.25), extent=ext(pss_fsca_samp_3class),
            cbar_ticks=[0.0, 0.5, 1.0],
            cbar_ticklabels=["No snow", "Uncertain", "Snow"],
        ),
        # fSCA
        dict(
            img=pss_fsca_samp.fsca.data,
            title="fSCA, 100 m grid\n(snow / (snow + no-snow))",
            cmap="Blues", clim=(0, 1), extent=ext(pss_fsca_samp),
            cbar_ticks=[0, 0.5, 1.0],
            cbar_ticklabels=["0%", "50%", "100%"], cbar_label="fSCA",
        ),
    ]

    fig, ax = plt.subplots(2, 3, figsize=(14, 10))
    fig.subplots_adjust(hspace=0.45)
    ax = ax.flatten()

    # Plot images
    for i, img_dict in enumerate(images):
        if img_dict['img'] is not None:
            plot_img(ax[i], **{k: v for k, v in img_dict.items() if k != 'img'},
                        img=img_dict['img'])
        else:
            ax[i].axis('off')

    # Flow arrows: RGB -> NDSI -> land cover -> fSCA -> binary SCA
    arrow_params = [
        (0, 1, (1.0, 0.5), (0.0, 0.5), "arc3"),                        # RGB -> NDSI
        (1, 2, (1.0, 0.5), (0.0, 0.5), "arc3"),                        # NDSI -> land cover
        (2, 5, (1.0, 0.5), (1.0, 0.5), "bar,angle=90,fraction=-0.2"),  # land cover -> fSCA
        (5, 4, (0.0, 0.5), (1.0, 0.5), "arc3"),                        # fSCA -> binary SCA
    ]
    for a_idx, b_idx, xyA, xyB, connstyle in arrow_params:
        con = matplotlib.patches.ConnectionPatch(
            xyA=xyA, xyB=xyB,
            coordsA="axes fraction", coordsB="axes fraction",
            axesA=ax[a_idx], axesB=ax[b_idx],
            shrinkA=1, shrinkB=1,
            ec="k", fc="k", linewidth=2, alpha=1,
            arrowstyle="-|>", connectionstyle=connstyle,
            mutation_scale=20,
        )
        ax[b_idx].add_artist(con)

    # Remove ticks
    for axis in ax:
        axis.set_xticks([])
        axis.set_yticks([])

    # Save figure
    fig.savefig(fig_file, dpi=300, bbox_inches='tight')
    print("Figure saved to:", fig_file)
    plt.close()

## Confusion matrices and recall time series

In [ ]:
# ----- Load consolidated confusion matrices, build tidy time-series table -----
# confusion_matrices.nc: vars TP/TN/FP/FN with dims (model, task, SWE_threshold_m, time)
cm_nc = xr.open_dataset(CM_FILE)

# Convert to a tidy DataFrame (one row per model/task/SWE/time)
ts_df = cm_nc[["TP", "TN", "FP", "FN"]].to_dataframe().reset_index()
ts_df = ts_df.rename(columns={"time": "datetime"})
ts_df["datetime"] = pd.to_datetime(ts_df["datetime"])
ts_df["SWE_threshold_m"] = ts_df["SWE_threshold_m"].astype(float)

# Drop rows that are entirely NaN counts (e.g. timestamps with no model match)
count_cols = ["TP", "TN", "FP", "FN"]
ts_df = ts_df.dropna(subset=count_cols, how="all").reset_index(drop=True)

# Per-timestamp metrics
denom_acc = ts_df["TP"] + ts_df["TN"] + ts_df["FP"] + ts_df["FN"]
denom_rec = ts_df["TP"] + ts_df["FN"]
denom_prec = ts_df["TP"] + ts_df["FP"]
ts_df["accuracy"] = (ts_df["TP"] + ts_df["TN"]).where(denom_acc > 0) / denom_acc
ts_df["recall"] = ts_df["TP"].where(denom_rec > 0) / denom_rec
ts_df["precision"] = ts_df["TP"].where(denom_prec > 0) / denom_prec

# ----- "Best" overall confusion matrix per task/model, selected across SWE thresholds -----
BEST_METRIC = 'accuracy'

cm_sum_df = (
    ts_df
    .groupby(['task', 'model', 'SWE_threshold_m'])[['TP', 'TN', 'FP', 'FN']]
    .sum()
    .reset_index()
)

denom_acc = cm_sum_df['TP'] + cm_sum_df['TN'] + cm_sum_df['FP'] + cm_sum_df['FN']
denom_rec = cm_sum_df['TP'] + cm_sum_df['FN']
denom_prec = cm_sum_df['TP'] + cm_sum_df['FP']

cm_sum_df['accuracy'] = (cm_sum_df['TP'] + cm_sum_df['TN']).where(denom_acc > 0) / denom_acc
cm_sum_df['recall'] = cm_sum_df['TP'].where(denom_rec > 0) / denom_rec
cm_sum_df['precision'] = cm_sum_df['TP'].where(denom_prec > 0) / denom_prec
denom_f1 = cm_sum_df['precision'] + cm_sum_df['recall']
cm_sum_df['f1'] = (2 * cm_sum_df['precision'] * cm_sum_df['recall']).where(denom_f1 > 0) / denom_f1

valid_groups = cm_sum_df.dropna(subset=[BEST_METRIC])
best_idx = valid_groups.groupby(['task', 'model'])[BEST_METRIC].idxmax()
cm_total_df = cm_sum_df.loc[best_idx].reset_index(drop=True)

all_groups = cm_sum_df[['task', 'model']].drop_duplicates()
missing_groups = all_groups.merge(
    cm_total_df[['task', 'model']], on=['task', 'model'], how='left', indicator=True
)
missing_groups = missing_groups[missing_groups['_merge'] == 'left_only'][['task', 'model']]
if not missing_groups.empty:
    print(f"WARNING: no usable SWE threshold found (all-NaN {BEST_METRIC}) for:\n{missing_groups}")
    cm_total_df = pd.concat([cm_total_df, missing_groups], ignore_index=True)

cm_total_df = cm_total_df.rename(columns={
    'SWE_threshold_m': 'best_SWE_threshold_m',
    BEST_METRIC: f'best_{BEST_METRIC}',
})

best_thresh_lookup = cm_total_df[['task', 'model', 'best_SWE_threshold_m']]
ts_best_df = ts_df.merge(best_thresh_lookup, on=['task', 'model'], how='inner')
ts_best_df = ts_best_df.loc[
    ts_best_df['SWE_threshold_m'] == ts_best_df['best_SWE_threshold_m']
].drop(columns=['best_SWE_threshold_m'])

recall_df = (
    ts_best_df
    .groupby(['task', 'datetime', 'model'])[['recall', 'precision']]
    .agg(['min', 'max', 'mean', 'std'])
    .reset_index()
)
recall_df.columns = ['_'.join(c).rstrip('_') for c in recall_df.columns]

cm_total_df

In [ ]:
### CONFUSION MATRICES ###

models = list(MODEL_NAMES)

def get_cm_counts(task, model, cm_total_df):
    row = cm_total_df.loc[(cm_total_df['task'] == task) & (cm_total_df['model'] == model)]
    if row.empty:
        return None
    vals = row[['TP', 'TN', 'FP', 'FN']].values[0]
    if np.any(pd.isna(vals)):
        return None
    tp, tn, fp, fn = vals
    return np.array(((tp, fn), (fp, tn)), dtype=float)


cm_fig_file = os.path.join(FIG_DIR, 'confusion_matrices_combined.png')

n_rows, n_cols = 3, len(models)
fig, ax = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 3.0 * n_rows))
ax = np.atleast_2d(ax)

# Diverging colormap for row 2 (Task 2 - Task 1)
diff_cmap = matplotlib.colormaps['RdBu']
diff_norm = matplotlib.colors.TwoSlopeNorm(vmin=-10, vcenter=0.0, vmax=10)
abs_norm = matplotlib.colors.Normalize(vmin=0, vmax=1)

for j, model in enumerate(models):
    counts_t1 = get_cm_counts(1, model, cm_total_df)
    counts_t2 = get_cm_counts(2, model, cm_total_df)

    # --- Row 0: Task 1 ---
    axr = ax[0, j]
    if counts_t1 is None:
        print(f"No Task 1 confusion matrix for {model}, skipping panel.")
        axr.remove()
    else:
        total1 = counts_t1.sum()
        frac1 = counts_t1 / total1 if total1 > 0 else counts_t1
        axr.matshow(frac1, cmap=recall_cmap, norm=abs_norm)
        for (r, c) in [(0, 0), (0, 1), (1, 0), (1, 1)]:
            val = counts_t1[r, c]
            pct = round(val / total1 * 100) if total1 > 0 else 0
            axr.text(c, r, f'{pct} %\n({int(val):,})', ha='center', va='center',
                        color='w', fontweight='bold', fontsize=8)
        axr.set_title(model, color=ds_colors_dict[model], fontweight='bold', pad=14)
        axr.xaxis.tick_top()
        axr.xaxis.set_label_position('top')

    # --- Row 1: Task 2 ---
    axr = ax[1, j]
    if counts_t2 is None:
        print(f"No Task 2 confusion matrix for {model}, skipping panel.")
        axr.remove()
    else:
        total2 = counts_t2.sum()
        frac2 = counts_t2 / total2 if total2 > 0 else counts_t2
        axr.matshow(frac2, cmap=recall_cmap, norm=abs_norm)
        for (r, c) in [(0, 0), (0, 1), (1, 0), (1, 1)]:
            val = counts_t2[r, c]
            pct = round(val / total2 * 100) if total2 > 0 else 0
            axr.text(c, r, f'{pct} %\n({int(val):,})', ha='center', va='center',
                        color='w', fontweight='bold', fontsize=8)
        axr.xaxis.set_ticks_position('none')  # avoid redundant top ticks mid-figure

    # --- Row 2: Task 2 - Task 1 (percentage-point difference) ---
    axr = ax[2, j]
    if counts_t1 is None or counts_t2 is None:
        print(f"Missing Task 1 or Task 2 CM for {model}, skipping difference panel.")
        axr.remove()
    else:
        total1, total2 = counts_t1.sum(), counts_t2.sum()
        frac1 = counts_t1 / total1 if total1 > 0 else np.zeros_like(counts_t1)
        frac2 = counts_t2 / total2 if total2 > 0 else np.zeros_like(counts_t2)
        diff_pct = (frac2 - frac1) * 100

        axr.matshow(diff_pct, cmap=diff_cmap, norm=diff_norm)
        for (r, c) in [(0, 0), (0, 1), (1, 0), (1, 1)]:
            val = diff_pct[r, c]
            sign = '+' if val >= 0 else ''
            axr.text(c, r, f'{sign}{val:.1f} %', ha='center', va='center',
                        color='k', fontweight='bold', fontsize=8)
        axr.xaxis.set_ticks_position('bottom')
        axr.xaxis.set_label_position('bottom')

    # x tick labels
    for r in (0, 2):
        if ax[r, j] in fig.axes:
            ax[r, j].set_xticks([0, 1])
            ax[r, j].set_xticklabels(['Snow', 'No snow'])
    if ax[1, j] in fig.axes:
        ax[1, j].set_xticks([])

    # y tick labels: only on leftmost column
    for r in range(n_rows):
        if ax[r, j] not in fig.axes:
            continue
        ax[r, j].set_yticks([0, 1])
        ax[r, j].set_yticklabels(['Snow', 'No snow'] if j == 0 else [])

# Row labels along the left margin
row_labels = ['Task 1', 'Task 2', r'Task 2 $-$ Task 1']
for axis, label in zip([ax[0,0], ax[1,0], ax[2,0]], row_labels):
    if axis in fig.axes:
        axis.set_ylabel(label, rotation=90, ha='center', va='center', fontweight='regular')
fig.text(0.005, 0.5, 'PlanetScope', rotation=90, ha='center', va='center', fontweight='bold')

# Colorbar for row 2
cax_diff = fig.add_axes([0.92, 0.125, 0.015, 0.20])
fig.colorbar(
    matplotlib.cm.ScalarMappable(cmap=diff_cmap, norm=diff_norm),
    cax=cax_diff, orientation='vertical', label=r'% change',
)

fig.savefig(cm_fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", cm_fig_file)
plt.close(fig)

In [ ]:
### RECALL TIME SERIES ###

recall_ts_fig_file = os.path.join(FIG_DIR, 'recall_timeseries_combined.png')

fig, ax = plt.subplots(3, 1, figsize=(12, 10))

# Dates at which to insert NaN "gap" rows so lines don't span long
# data-free periods (e.g. summer). Adjust to match your actual gaps.
gap_dates = [pd.Timestamp('2023-08-01'), pd.Timestamp('2024-08-01')]

def _with_gaps(df, model, value_cols):
    gap_rows = pd.DataFrame({
        'datetime': gap_dates,
        'model': [model] * len(gap_dates),
        **{c: [np.nan] * len(gap_dates) for c in value_cols},
    })
    out = pd.concat([df, gap_rows], ignore_index=True)
    return out.sort_values(by='datetime').reset_index(drop=True)

# --- Rows 0-1: Task 1 and Task 2 recall time series ---
for row, task in zip([0, 1], [1, 2]):
    recall_task_df = recall_df.loc[recall_df['task'] == task]
    for model in models:
        m_df = recall_task_df.loc[recall_task_df['model'] == model]
        if m_df.empty:
            continue
        m_df = _with_gaps(m_df, model, ['recall_min', 'recall_max', 'recall_mean', 'recall_std'])
        ax[row].fill_between(
            m_df['datetime'], m_df['recall_min'], m_df['recall_max'],
            color=ds_colors_dict[model], alpha=0.3
        )
        ax[row].plot(
            m_df['datetime'], m_df['recall_mean'], '.-',
            color=ds_colors_dict[model], label=model
        )
    ax[row].set_ylabel('Recall')
    ax[row].set_ylim(0, 1)
    ax[row].grid(alpha=0.3)

ax[0].legend(loc='upper center', ncols=len(models), bbox_to_anchor=[0.4, 1.25, 0.2, 0.2])

# --- Row 2: Change in recall (Task 2 - Task 1) ---
# Match Task 1 and Task 2 by acquisition datetime
r1 = recall_df.loc[recall_df['task'] == 1, ['datetime', 'model', 'recall_mean']]
r2 = recall_df.loc[recall_df['task'] == 2, ['datetime', 'model', 'recall_mean']]
recall_diff_df = r1.merge(r2, on=['datetime', 'model'], suffixes=('_t1', '_t2'), how='inner')

if recall_diff_df.empty:
    print("WARNING: no overlapping (datetime, model) pairs found between "
            "Task 1 and Task 2 -- cannot compute a recall change time "
            "series by exact date. Consider aligning by day-of-year instead.")
else:
    recall_diff_df['recall_diff'] = recall_diff_df['recall_mean_t2'] - recall_diff_df['recall_mean_t1']
    for model in models:
        d_df = recall_diff_df.loc[recall_diff_df['model'] == model][['datetime', 'recall_diff']].copy()
        if d_df.empty:
            continue
        d_df = _with_gaps(d_df, model, ['recall_diff'])
        ax[2].plot(
            d_df['datetime'], d_df['recall_diff'], '.-',
            color=ds_colors_dict[model], label=model
        )

ax[2].axhline(0, color='k', linewidth=0.8, linestyle='--')
ax[2].set_xlabel('Date')
ax[2].set_ylabel(r'$\Delta$ Recall')
ax[2].set_ylim(-1, 1)
ax[2].grid(alpha=0.3)

# Row labels tied to each subplot's own axes coordinates
row_labels = ['Task 1', 'Task 2', r'Task 2 $-$ Task 1']
for r, label in enumerate(row_labels):
    ax[r].text(
        -0.11, 0.5, label, transform=ax[r].transAxes, rotation=90,
        ha='center', va='center', fontsize=11, fontweight='regular'
        )

fig.savefig(recall_ts_fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", recall_ts_fig_file)
plt.close(fig)

## Recall with terrain

In [ ]:
def plot_recall_polar(terrain_bin_da, cmap, cmap_norm, ax=None):

    # Load the dataset
    elev_bin_edges = terrain_bin_da.attrs['elev_bins']
    aspect_bin_edges = terrain_bin_da.attrs['aspect_bins']

    # Create the axis if needed
    if ax==None:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='polar')

    ax.set_theta_zero_location('N')     # top = N
    ax.set_theta_direction(-1)          # Clockwise

    # Get bin centers and widths for elevation and aspect
    elev_bin_width = elev_bin_edges[1] - elev_bin_edges[0]
    elev_centers = (elev_bin_edges[1:] + elev_bin_edges[:-1]) / 2
    aspect_bin_width = aspect_bin_edges[1] - aspect_bin_edges[0]
    aspect_centers = (aspect_bin_edges[:-1] + aspect_bin_edges[1:]) / 2
    theta_centers = np.deg2rad(aspect_centers)

    # Iterate over elevation and aspect bin centers
    for i, elev in enumerate(elev_centers):
        for j, theta in enumerate(theta_centers):
            val = terrain_bin_da.data[i, j]
            color = 'whitesmoke' if np.isnan(val) else cmap(cmap_norm(val))
            ax.bar(
                x=theta,
                height=elev_bin_width,
                bottom=elev_bin_edges[i],
                width=np.deg2rad(aspect_bin_width),
                color=color,
                edgecolor='none',
                align='center'
            )

    ax.set_xticks(np.deg2rad(np.arange(0,360,45)))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    ax.set_ylim(500, 2500)
    ax.set_yticks(np.arange(1000, 3000, 500))
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
    ax.grid(color='k', alpha=0.3)

    return


# Open the consolidated recall dataset once.
# recall var dims: (model, task, SWE_threshold_m, time, elev_bin, aspect_bin)
# elev_bins / aspect_bins are stored as attrs on the "recall" variable.
recall_nc = xr.open_dataset(RECALL_FILE)
RECALL_ELEV_BINS = np.asarray(recall_nc["recall"].attrs["elev_bins"])
RECALL_ASPECT_BINS = np.asarray(recall_nc["recall"].attrs["aspect_bins"])


def load_best_terrain_recall(task, model, cm_total_df, recall_nc):
    """
    Select the terrain-binned recall array for the SINGLE SWE threshold chosen
    as "best" for (task, model) in cm_total_df, averaged over time.
    Returns None (and prints why) if unavailable.
    """
    best_row = cm_total_df.loc[
        (cm_total_df['task'] == task) & (cm_total_df['model'] == model)
    ]
    if best_row.empty or best_row['best_SWE_threshold_m'].isna().all():
        print(f"No best SWE threshold available for {model} Task {task}, skipping.")
        return None
    best_swe = float(best_row['best_SWE_threshold_m'].values[0])

    if model not in recall_nc["model"].values:
        print(f"{model} not in recall dataset, skipping.")
        return None

    # Nearest match on the SWE coordinate (guards against float rounding)
    try:
        da = recall_nc["recall"].sel(
            model=model, task=task, SWE_threshold_m=best_swe, method=None
        )
    except KeyError:
        da = recall_nc["recall"].sel(model=model, task=task).sel(
            SWE_threshold_m=best_swe, method="nearest"
        )

    # dims now: (time, elev_bin, aspect_bin) -> average over time
    if np.all(np.isnan(da.values)):
        print(f"All-NaN recall for {model} Task {task} at SWE {best_swe} m, skipping.")
        return None

    terrain_bin_tmean_da = da.mean(dim="time")

    # Restore elev_bins/aspect_bins attrs (dropped during .mean())
    terrain_bin_tmean_da = terrain_bin_tmean_da.assign_attrs({
        "elev_bins": RECALL_ELEV_BINS,
        "aspect_bins": RECALL_ASPECT_BINS,
    })
    return terrain_bin_tmean_da


# Combined terrain recall figure:
# - Row 1: Task 1
# - Row 2: Task 2
# - Row 3: Task 2 - Task 1
models = list(MODEL_NAMES)
tasks = [1, 2]

fig_file = os.path.join(FIG_DIR, "terrain_recall_combined.png")

# Load all (task, model) terrain recall arrays first
recall_terrain_cache = {}
for task in tasks:
    for model in models:
        recall_terrain_cache[(task, model)] = load_best_terrain_recall(
            task, model, cm_total_df, recall_nc
        )

# Diverging colormap/norm for row 3 (recall change)
recall_diff_cmap = matplotlib.colormaps['RdBu']
recall_diff_norm = matplotlib.colors.TwoSlopeNorm(vmin=-0.5, vcenter=0.0, vmax=0.5)

n_rows, n_cols = 3, len(models)
fig, ax = plt.subplots(
    n_rows, n_cols, figsize=(3.2 * n_cols, 3.2 * n_rows),
    subplot_kw={"projection": "polar"}
)
ax = np.atleast_2d(ax)

for j, model in enumerate(models):
    da_t1 = recall_terrain_cache.get((1, model))
    da_t2 = recall_terrain_cache.get((2, model))

    # --- Row 0: Task 1 ---
    if da_t1 is None:
        ax[0, j].remove()
    else:
        plot_recall_polar(da_t1, cmap=recall_cmap, cmap_norm=recall_cmap_norm, ax=ax[0, j])

    # --- Row 1: Task 2 ---
    if da_t2 is None:
        ax[1, j].remove()
    else:
        plot_recall_polar(da_t2, cmap=recall_cmap, cmap_norm=recall_cmap_norm, ax=ax[1, j])

    # Column title (model name) above row 0 only
    if da_t1 is not None or da_t2 is not None:
        title_ax = ax[0, j] if da_t1 is not None else ax[1, j]
        title_ax.set_title(model, color=ds_colors_dict[model], fontweight='bold', pad=12)

    # --- Row 2: Task 2 - Task 1 ---
    if da_t1 is None or da_t2 is None:
        print(f"Missing Task 1 or Task 2 terrain recall for {model}, skipping change plot.")
        ax[2, j].remove()
        continue

    recall_change_da = da_t2 - da_t1
    recall_change_da = recall_change_da.assign_attrs(dict(da_t1.attrs))

    plot_recall_polar(recall_change_da, cmap=recall_diff_cmap, cmap_norm=recall_diff_norm, ax=ax[2, j])

# Row labels along the left margin
row_labels = ['Task 1', 'Task 2', r'Task 2 $-$ Task 1']
for r, label in enumerate(row_labels):
    fig.text(
        0.06, 1 - (r + 0.5) / n_rows, label,
        rotation=90, ha='center', va='center', fontsize=12, fontweight='regular'
    )

# Colorbar for rows 1-2 (TOTAL RECALL)
cax_recall = fig.add_axes([0.95, 0.4, 0.015, 0.45])
fig.colorbar(
    matplotlib.cm.ScalarMappable(cmap=recall_cmap, norm=recall_cmap_norm),
    cax=cax_recall, orientation='vertical', label='Recall',
)

# Colorbar for row 3 (recall change)
cax_diff = fig.add_axes([0.95, 0.12, 0.015, 0.20])
fig.colorbar(
    matplotlib.cm.ScalarMappable(cmap=recall_diff_cmap, norm=recall_diff_norm),
    cax=cax_diff, orientation='vertical', label=r'Recall change',
)

fig.savefig(fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", fig_file)
plt.close(fig)

## Lidar coverage polar plot

In [ ]:
import xrspatial

lidar_domain_file = '/Users/rdcrlrka/Research/SkySat-Stereo/study-sites/MCS/SNEX_MCS_Lidar/SNEX_MCS_Lidar_20250501_DTM_V01.0.tif'
dem_file = "/Users/rdcrlrka/Research/SkySat-Stereo/study-sites/MCS/refdem/USGS_3DEP/MCS_USGS_3DEP_merged.tif"
target_grid_file = os.path.join(OUT_DIR, "target_grid.tif")
CRS = "EPSG:32611"

with (
    rxr.open_rasterio(lidar_domain_file, masked=True).squeeze() as lidar_da,
    rxr.open_rasterio(target_grid_file, masked=True).squeeze() as target_grid_da,
    rxr.open_rasterio(dem_file, masked=True).squeeze() as dem_da,
):
    coverage_mask = xr.where(np.isnan(lidar_da), 0, 1).astype('float32')
    coverage_mask = coverage_mask.rio.write_crs(CRS)
    coverage_mask = coverage_mask.rio.write_nodata(0)

    # Reproject coverage mask
    coverage_mask_regrid = coverage_mask.rio.reproject_match(
        target_grid_da, resampling='nearest'
    )
    coverage_mask_regrid = xr.where(np.isnan(coverage_mask_regrid), 0, coverage_mask_regrid)

    # Reproject DEM
    dem_regrid = dem_da.rio.reproject_match(target_grid_da, resampling='bilinear')
    aspect_regrid = xrspatial.aspect(dem_regrid)

# Reuse the same bin edges as the recall polar plots
elev_bins = da_t2.attrs['elev_bins']
aspect_bins = da_t2.attrs['aspect_bins']


def bin_coverage_by_terrain(coverage_da, elev_da, aspect_da, elev_bins, aspect_bins):
    elev_bin_idx = np.digitize(elev_da.values, elev_bins) - 1
    aspect_bin_idx = np.digitize(aspect_da.values, aspect_bins) - 1
    n_elev_bins = len(elev_bins) - 1
    n_aspect_bins = len(aspect_bins) - 1

    cov = coverage_da.values
    coverage_frac = np.full((n_elev_bins, n_aspect_bins), np.nan)
    for j in range(n_elev_bins):
        for k in range(n_aspect_bins):
            mask = (elev_bin_idx == j) & (aspect_bin_idx == k)
            if np.sum(mask) > 0:
                coverage_frac[j, k] = np.nanmean(cov[mask])

    coverage_frac_da = xr.DataArray(
        coverage_frac,
        dims=['elev_bin', 'aspect_bin'],
        coords={'elev_bin': np.arange(n_elev_bins), 'aspect_bin': np.arange(n_aspect_bins)},
        name='lidar_coverage_frac',
    )
    coverage_frac_da = coverage_frac_da.assign_attrs({'elev_bins': elev_bins, 'aspect_bins': aspect_bins})
    return coverage_frac_da


lidar_coverage_frac_da = bin_coverage_by_terrain(
    coverage_mask_regrid, dem_regrid, aspect_regrid, elev_bins, aspect_bins
)


def plot_coverage_polar(
        coverage_frac_da, ax=None, coverage_threshold=0.0, 
        coverage_color='#ec7014', no_coverage_color="#d8d8d8"
        ):
    elev_bin_edges = coverage_frac_da.attrs['elev_bins']
    aspect_bin_edges = coverage_frac_da.attrs['aspect_bins']

    if ax is None:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='polar')

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)

    elev_bin_width = elev_bin_edges[1] - elev_bin_edges[0]
    elev_centers = (elev_bin_edges[1:] + elev_bin_edges[:-1]) / 2
    aspect_bin_width = aspect_bin_edges[1] - aspect_bin_edges[0]
    aspect_centers = (aspect_bin_edges[:-1] + aspect_bin_edges[1:]) / 2
    theta_centers = np.deg2rad(aspect_centers)

    for i, elev in enumerate(elev_centers):
        for j, theta in enumerate(theta_centers):
            val = coverage_frac_da.data[i, j]
            has_coverage = (not np.isnan(val)) and (val > coverage_threshold)
            ax.bar(
                x=theta,
                height=elev_bin_width,
                bottom=elev_bin_edges[i],
                width=np.deg2rad(aspect_bin_width),
                color=coverage_color if has_coverage else no_coverage_color,
                edgecolor='none',
                align='center',
            )

    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45)))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    ax.set_ylim(500, 2500)
    ax.set_yticks(np.arange(1000, 3000, 500))
    ax.grid(color='k', alpha=0.3)

    return ax


fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='polar')
plot_coverage_polar(lidar_coverage_frac_da, ax=ax)
ax.set_title('Lidar Coverage')
fig.tight_layout()

lidar_fig_file = os.path.join(FIG_DIR, 'lidar_coverage_polar.png')
fig.savefig(lidar_fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", lidar_fig_file)
plt.close()